# Module 03 — Hands-On Notebook
## Building a Misinformation Classifier

**Course:** AI for Social Good | **Module:** 03 — AI & Misinformation

In this notebook we build a working text classifier that can distinguish
reliable from unreliable statements — and critically examine its limits.

---

## Setup

In [ ]:
!pip install datasets -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.pipeline import Pipeline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi']=110
print('Ready.')

## 1. Load Data

We use the LIAR dataset — 12,800 labelled political statements.
We simplify 6 labels to binary: Fake (0) vs Real (1).

In [ ]:
raw = load_dataset('liar')

def to_df(split):
    df = pd.DataFrame(raw[split])
    fake = {'pants-fire','false','barely-true'}
    df['label_name'] = df['label'].apply(lambda x: raw[split].features['label'].int2str(x))
    df['binary'] = df['label_name'].apply(lambda x: 0 if x in fake else 1)
    df['binary_name'] = df['binary'].map({0:'Fake',1:'Real'})
    return df

train_df = to_df('train')
test_df  = to_df('test')
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')
train_df[['statement','label_name','binary_name']].head(5)

## 2. Text Preprocessing

In [ ]:
def preprocess(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

train_df['clean'] = train_df['statement'].apply(preprocess)
test_df['clean']  = test_df['statement'].apply(preprocess)
print('Sample:')
print(train_df[['statement','clean']].head(3))

## 3. Train the Classifier

We use a **Pipeline** — TF-IDF vectorisation followed by Logistic Regression.
This is the standard pattern for text classification in scikit-learn.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=8000, ngram_range=(1,2),
                               stop_words='english', min_df=3)),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(train_df['clean'], train_df['binary'])
y_pred = pipeline.predict(test_df['clean'])

print('=== Classification Report ===')
print(classification_report(test_df['binary'], y_pred, target_names=['Fake','Real']))

## 4. Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(test_df['binary'], y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Fake','Real']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold')

vect = pipeline.named_steps['tfidf']
clf  = pipeline.named_steps['clf']
features = vect.get_feature_names_out()
coefs    = clf.coef_[0]
top_fake = np.argsort(coefs)[:15]
top_real = np.argsort(coefs)[-15:][::-1]
words  = [features[i] for i in top_fake]
values = [abs(coefs[i]) for i in top_fake]
axes[1].barh(words[::-1], values[::-1], color='#e74c3c', edgecolor='white')
axes[1].set_title('Top 15 Fake Indicators', fontweight='bold')
axes[1].set_xlabel('Coefficient magnitude')
plt.tight_layout()
plt.show()

## 5. Live Prediction Demo

In [ ]:
def predict(text):
    clean = preprocess(text)
    pred  = pipeline.predict([clean])[0]
    prob  = pipeline.predict_proba([clean])[0]
    label = 'REAL' if pred == 1 else 'FAKE'
    print(f'[{label}] Fake={prob[0]:.1%} Real={prob[1]:.1%} — "{text}"')

predict('The government has invested heavily in rural health centres this year.')
predict('Scientists confirm that drinking hot water cures all known diseases.')
predict('Unemployment has fallen to record lows according to official statistics.')

## 6. Critical Reflection — Required

This is the most important part of the notebook. A model that gets 68% accuracy
on an English dataset will not work the same way on WhatsApp messages in Camfranglais.

Answer these questions:

1. Look at the top 15 fake indicators. Do any of those words surprise you? Why?

2. What would happen if we deployed this model to flag messages in a Cameroonian
   WhatsApp group? List at least three specific failure modes.

3. Who would be most harmed by false positives (legitimate content flagged as fake)?

4. **Extension:** Try Naive Bayes (`from sklearn.naive_bayes import MultinomialNB`)
   as the classifier. Compare its F1-score to Logistic Regression.

In [ ]:
# Your reflections and extension work here
